# Phase 6 — Review Score Prediction (Binary Classification with Random Forest)

**Goal:** Predict whether a customer will give a **low review score** (1 or 2 stars) based on order features available at the time of delivery.

**Business use case:** E-commerce platforms can use this model to:
- **Proactively intervene** before a bad review is posted — trigger a customer service call or offer a discount
- **Route high-risk orders** to premium delivery partners
- **Prioritise complaints** in the support queue

**Type of problem:** Binary Classification
- Target: `is_low_review` → 1 if review score ≤ 2, else 0
- Features: delivery delay, order value, number of items, product category, payment method, seller state

**Model: Random Forest Classifier**
An ensemble of decision trees trained on random subsets of data and features. Each tree votes, and the majority wins. Key properties:
- Handles mixed feature types well
- Robust to outliers
- Built-in feature importance
- Usually needs minimal tuning to perform well

## 0. Setup & Connect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.titlesize': 14})

load_dotenv()
engine = create_engine(os.getenv('SUPABASE_URL'))
print('Connected to Supabase')

## 1. Build the Feature Matrix via SQL

We build one wide analytical table by joining multiple tables in a **single SQL query**. This is more efficient than loading all tables separately and merging in pandas.

**Features selected:**
- `delivery_delay_days`: actual delivery - estimated delivery (key predictor — late = bad review)
- `order_value`: total amount paid by customer
- `freight_pct`: freight cost as % of order value (high freight → customer frustration)
- `item_count`: number of items in the order
- `payment_type`: credit_card, boleto, voucher, debit_card
- `product_category`: what they bought
- `seller_state`: where the seller is located (affects delivery time)

In [ ]:
query = """
WITH order_features AS (
    SELECT
        o.order_id,
        r.review_score,
        -- Delivery delay in days (positive = late, negative = early)
        EXTRACT(EPOCH FROM (
            o.order_delivered_customer_date::timestamp - 
            o.order_estimated_delivery_date::timestamp
        )) / 86400.0                                        AS delivery_delay_days,
        -- Order financials
        SUM(oi.price)                                       AS product_value,
        SUM(oi.freight_value)                               AS freight_value,
        SUM(oi.price + oi.freight_value)                   AS order_value,
        COUNT(oi.order_item_id)                             AS item_count,
        -- Payment info
        MAX(p.payment_type)                                 AS payment_type,
        MAX(p.payment_installments)                        AS payment_installments,
        -- Product category (most expensive item in order)
        COALESCE(MAX(ct.product_category_name_english), 'unknown') AS product_category,
        -- Seller location
        MAX(s.seller_state)                                AS seller_state
    FROM orders o
    JOIN order_items oi   ON o.order_id = oi.order_id
    LEFT JOIN reviews r   ON o.order_id = r.order_id
    LEFT JOIN payments p  ON o.order_id = p.order_id
    LEFT JOIN products pr ON oi.product_id = pr.product_id
    LEFT JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
    LEFT JOIN sellers s   ON oi.seller_id = s.seller_id
    WHERE 
        o.order_status = 'delivered'
        AND o.order_delivered_customer_date IS NOT NULL
        AND o.order_estimated_delivery_date IS NOT NULL
        AND r.review_score IS NOT NULL
    GROUP BY o.order_id, r.review_score, o.order_delivered_customer_date, o.order_estimated_delivery_date
)
SELECT *,
    -- Freight as percentage of order value
    CASE WHEN order_value > 0 THEN freight_value / order_value ELSE 0 END AS freight_pct
FROM order_features
"""

df = pd.read_sql(query, engine)
print(f'Loaded {len(df):,} orders with features')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Create the Target Variable

**Binary classification target:** `is_low_review`
- 1 = review score of 1 or 2 (unhappy customer)
- 0 = review score of 3, 4, or 5 (acceptable / happy)

**Class imbalance:** In Olist, ~80%+ of reviews are 4-5 stars. This means our dataset is **imbalanced** — there are far fewer low-review orders than high-review ones.

If we trained on imbalanced data naively, the model could achieve 80%+ accuracy by just predicting 'not low review' every time — which is useless. We'll use `class_weight='balanced'` in RandomForest to handle this.

In [ ]:
# Create binary target
df['is_low_review'] = (df['review_score'] <= 2).astype(int)

# Class distribution
class_dist = df['is_low_review'].value_counts()
print('Target class distribution:')
print(f'  High review (0): {class_dist[0]:,} orders ({class_dist[0]/len(df)*100:.1f}%)')
print(f'  Low review  (1): {class_dist[1]:,} orders ({class_dist[1]/len(df)*100:.1f}%)')
print(f'  Imbalance ratio: {class_dist[0]/class_dist[1]:.1f}x more high reviews')

# Review score distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

review_counts = df['review_score'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(review_counts.index, review_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Review Score Distribution (1-5 stars)')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Count')
for i, (score, count) in enumerate(review_counts.items()):
    axes[0].text(score, count + 100, f'{count:,}', ha='center', fontsize=9)

axes[1].pie([class_dist[0], class_dist[1]], 
           labels=['High Review (3-5★)', 'Low Review (1-2★)'],
           colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', 
           startangle=90, pctdistance=0.75)
axes[1].set_title('Binary Target: Low vs High Review')

plt.suptitle('Class Distribution Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

**Label Encoding** for categorical features: Random Forest in scikit-learn cannot handle string values directly. We convert categorical strings to integer codes.

For example: `['boleto', 'credit_card', 'debit_card', 'voucher']` → `[0, 1, 2, 3]`

**Important caveat:** LabelEncoder implies an ordinal relationship (0 < 1 < 2) which may not exist. For high-cardinality features or linear models, **One-Hot Encoding** would be better. Random Forest can handle LabelEncoded categoricals well because it makes binary splits.

In [ ]:
# Select features for model
feature_cols = [
    'delivery_delay_days',
    'order_value',
    'freight_pct',
    'item_count',
    'payment_installments',
    'payment_type',
    'product_category',
    'seller_state'
]

df_model = df[feature_cols + ['is_low_review']].copy()

# Drop nulls
df_model = df_model.dropna()
print(f'Rows after dropping nulls: {len(df_model):,} (dropped {len(df)-len(df_model):,})')

# Label-encode categorical features
categorical_cols = ['payment_type', 'product_category', 'seller_state']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le
    print(f'  Encoded "{col}": {len(le.classes_)} unique values')

# Feature matrix and target vector
X = df_model[feature_cols]
y = df_model['is_low_review']

print(f'\nFinal feature matrix shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')

## 4. Train/Test Split & Model Training

Unlike time series, here rows are **independent** (one order does not affect another), so **random stratified splitting** is correct.

`stratify=y` ensures the class ratio (low/high review) is preserved in both train and test sets — important for imbalanced datasets.

`class_weight='balanced'` automatically adjusts sample weights inversely proportional to class frequencies. This tells the model to pay more attention to the rare class (low reviews).

In [ ]:
# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y          # preserves class ratio in both sets
)

print(f'Training set: {len(X_train):,} orders')
print(f'Test set:     {len(X_test):,} orders')
print(f'Train class ratio: {y_train.mean():.3f} (low review rate)')
print(f'Test  class ratio: {y_test.mean():.3f} (should match train)')

# Train Random Forest with class balancing
rf_model = RandomForestClassifier(
    n_estimators=200,          # 200 trees
    max_depth=12,              # prevents overfitting
    min_samples_leaf=10,       # each leaf needs ≥10 samples
    class_weight='balanced',   # handles class imbalance
    random_state=42,
    n_jobs=-1                  # use all CPU cores
)

print('\nTraining Random Forest...')
rf_model.fit(X_train, y_train)
print('Training complete!')

## 5. Evaluate Model Performance

**For imbalanced classification, accuracy alone is misleading.** Use these metrics:

- **Precision** (for class 1): Of all predicted low-review orders, what % were actually low-review? (Minimises false alarms)
- **Recall** (for class 1): Of all actual low-review orders, what % did we catch? (Minimises misses)
- **F1 Score**: Harmonic mean of precision and recall. The go-to metric for imbalanced classification.
- **ROC-AUC**: Area under the ROC curve. 0.5 = random, 1.0 = perfect. Threshold-independent — measures separability between classes.

**Business trade-off:** High recall = we catch more unhappy customers but send unnecessary interventions. High precision = we only intervene when confident, but miss some unhappy customers. The right trade-off depends on the cost of intervention vs cost of a lost customer.

In [ ]:
# Generate predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]  # probability of low review

# Core metrics
print('=' * 55)
print('RANDOM FOREST CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['High Review (0)', 'Low Review (1)']))

roc_auc = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC:  {roc_auc:.4f}')
print('\nNote: Focus on Low Review (1) metrics — that\'s our business target.')

## 6. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: High', 'Pred: Low'],
            yticklabels=['Actual: High', 'Actual: Low'])
axes[0].set_title(f'Confusion Matrix\nAccuracy: {accuracy:.3f}')

# Add percentages as subtitle
tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')

# --- ROC Curve ---
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'Random Forest (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier (AUC = 0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Recall/Sensitivity)')
axes[1].set_title(f'ROC Curve\nROC-AUC = {roc_auc:.4f}')
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.suptitle('Model Evaluation — Random Forest Classifier', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nReading the confusion matrix:')
print(f'  True Negatives  (correctly predicted high review): {tn:,}')
print(f'  False Positives (incorrectly flagged as low review): {fp:,}')
print(f'  False Negatives (missed actual low reviews): {fn:,}  ← COSTLY')
print(f'  True Positives  (correctly caught low reviews): {tp:,}')

## 7. Feature Importance — What Predicts Bad Reviews?

Random Forest computes **feature importance** as the average reduction in impurity (Gini impurity for classification) contributed by each feature across all trees.

This is one of the most useful outputs — it tells you **which factors most strongly predict a bad review**. These are actionable insights for the business.

In [ ]:
# Feature importances
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if imp > importances.quantile(0.75) else '#3498db' for imp in importances.values]
bars = plt.barh(importances.index, importances.values, color=colors, edgecolor='white')
plt.xlabel('Feature Importance (Mean Decrease in Gini Impurity)')
plt.title('Random Forest Feature Importance\nWhat Predicts a Low Review Score?', fontweight='bold')

# Add value labels
for bar, val in zip(bars, importances.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Top 3 most important features:')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {imp:.4f}')
    
print('\nBusiness interpretation:')
print('  → If delivery_delay_days is #1: Late deliveries drive bad reviews (expected!)')
print('  → If order_value is high: Expensive orders get scrutinised more')
print('  → Category/seller_state: Some categories/regions have systematic issues')

## 8. Cross-Validation (Robust Evaluation)

A single train/test split can be lucky or unlucky depending on which rows ended up in the test set. **5-Fold Cross-Validation** splits the data into 5 equal folds, trains on 4 and tests on 1, rotating through all 5 combinations. This gives 5 scores whose mean and std are more reliable than a single split.

In [ ]:
print('Running 5-fold cross-validation (this may take 1-2 minutes)...')

cv_rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight='balanced', 
    random_state=42, n_jobs=-1
)

# ROC-AUC scores across 5 folds
cv_scores = cross_val_score(cv_rf, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'\n5-Fold Cross-Validation ROC-AUC Scores:')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'Mean AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'\nThis is the most reliable estimate of model performance on unseen data.')

## 9. Interview Talking Points

1. **Why binary classification instead of predicting the exact score (1-5)?** Predicting all 5 scores is a multi-class problem. Business decisions are usually binary: intervene or don't. Collapsing to binary simplifies the problem and improves model performance since rare classes (1 and 2) gain more signal when grouped.

2. **Why not just use accuracy?** Class imbalance. If 85% of orders get high reviews, a dumb model that always predicts 'high' gets 85% accuracy — useless for catching unhappy customers. ROC-AUC and F1 for the minority class are much better metrics.

3. **What is `class_weight='balanced'`?** It makes the algorithm weight each sample by `n_samples / (n_classes * n_samples_in_class)`. Rare class samples are weighted more heavily during training, forcing the model to learn patterns from them.

4. **What does ROC-AUC tell you?** The probability that a randomly chosen positive instance (low review) has a higher predicted score than a randomly chosen negative instance (high review). 0.5 = random, 1.0 = perfect. It's threshold-independent — works across all decision thresholds.

5. **How would you deploy this?** Save the model with `joblib.dump()`. Build a REST API with FastAPI that accepts order features and returns a risk score (0-1). Set a threshold (e.g., 0.4) above which an alert is triggered in the customer service system.